# BÀI TẬP: E-COMMERCE DATA (ONLINE RETAIL)
**Nguồn:** kaggle.com/datasets/carrie1/ecommerce-data (541,909 dòng)


## Setup

In [2]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
from scipy import stats

sns.set_style('whitegrid')

csv_path = 'https://raw.githubusercontent.com/databricks/Spark-The-Definitive-Guide/master/data/retail-data/all/online-retail-dataset.csv'

df = pd.read_csv(csv_path, encoding='ISO-8859-1', on_bad_lines='skip')
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
print('Loaded from:', csv_path)
df.head()

Loaded from: https://raw.githubusercontent.com/databricks/Spark-The-Definitive-Guide/master/data/retail-data/all/online-retail-dataset.csv


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


---
# PHẦN A — DATA PROFILING
## A.1. Data size, column names, data types

In [3]:
# TODO
print("Kích thước dữ liệu (rows, columns):", df.shape)
print("\nDanh sách các cột và kiểu dữ liệu:")
print(df.dtypes)
print("\nThông tin chi tiết:")
df.info()

Kích thước dữ liệu (rows, columns): (541909, 8)

Danh sách các cột và kiểu dữ liệu:
InvoiceNo                 str
StockCode                 str
Description               str
Quantity                int64
InvoiceDate    datetime64[us]
UnitPrice             float64
CustomerID            float64
Country                   str
dtype: object

Thông tin chi tiết:
<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  str           
 1   StockCode    541909 non-null  str           
 2   Description  540455 non-null  str           
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[us]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  str           
dtypes: datetime64[us](1), float64(2), int

## A.2. Missing values & Duplicate data

In [4]:
# TODO
print(df.isnull().sum())

duplicate_count = df.duplicated().sum()

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64


## A.3. Invalid values

In [ ]:
# TODO
print(df[['Quantity', 'UnitPrice']].describe())

print(f"\nSố dòng có Quantity <= 0 : {(df['Quantity'] <= 0).sum()}")

            Quantity      UnitPrice
count  541909.000000  541909.000000
mean        9.552250       4.611114
std       218.081158      96.759853
min    -80995.000000  -11062.060000
25%         1.000000       1.250000
50%         3.000000       2.080000
75%        10.000000       4.130000
max     80995.000000   38970.000000

Số dòng có Quantity <= 0 (hàng trả lại/hủy): 10624


## A.4. Create a new column
Làm sạch dữ liệu (loại Quantity<=0, UnitPrice<=0), tạo cột `Sales` = Quantity * UnitPrice.

In [12]:
# TODO
df.loc[(df['Quantity'] <= 0) & (df['UnitPrice'] <= 0)]

df['Sales']= df['Quantity'] * df['UnitPrice']
df['Sales'].head()

0    15.30
1    20.34
2    22.00
3    20.34
4    20.34
Name: Sales, dtype: float64

---
# PHẦN B — DESCRIPTIVE STATISTICS
## Group 1 — Central Tendency

In [14]:
# TODO
cols = ['Quantity', 'UnitPrice', 'Sales']

print("Giá trị trung bình (Mean):")
print(df[cols].mean())

print("\nGiá trị trung vị (Median):")
print(df[cols].median())

print("\nGiá trị xuất hiện nhiều nhất :")
print(df[cols].mode())

Giá trị trung bình (Mean):
Quantity      9.552250
UnitPrice     4.611114
Sales        17.987795
dtype: float64

Giá trị trung vị (Median):
Quantity     3.00
UnitPrice    2.08
Sales        9.75
dtype: float64

Giá trị xuất hiện nhiều nhất :
   Quantity  UnitPrice  Sales
0         1       1.25   15.0


## Group 2 — Dispersion

In [15]:
# TODO
print("Độ lệch chuẩn (Standard Deviation):")
print(df[cols].std())

print("\nPhương sai (Variance):")
print(df[cols].var())

print("\nKhoảng biến thiên (Range = Max - Min):")
print(df[cols].max() - df[cols].min())

print("\nKhoảng tứ phân vị (IQR = Q3 - Q1):")
print(df[cols].quantile(0.75) - df[cols].quantile(0.25))

Độ lệch chuẩn (Standard Deviation):
Quantity     218.081158
UnitPrice     96.759853
Sales        378.810824
dtype: float64

Phương sai (Variance):
Quantity      47559.391409
UnitPrice      9362.469164
Sales        143497.640005
dtype: float64

Khoảng biến thiên (Range = Max - Min):
Quantity     161990.00
UnitPrice     50032.06
Sales        336939.20
dtype: float64

Khoảng tứ phân vị (IQR = Q3 - Q1):
Quantity      9.00
UnitPrice     2.88
Sales        14.00
dtype: float64


## Group 3 — Location and Shape

In [16]:
# TODO
print("Độ lệch (Skewness):")
print(df[cols].skew())

print("\nĐộ nhọn (Kurtosis):")
print(df[cols].kurt())

print("\nCác mức phân vị (Q1=25%, Q2=50%, Q3=75%):")
print(df[cols].quantile([0.25, 0.5, 0.75]))

Độ lệch (Skewness):
Quantity      -0.264076
UnitPrice    186.506972
Sales         -0.964389
dtype: float64

Độ nhọn (Kurtosis):
Quantity     119769.160031
UnitPrice     59005.719097
Sales        151197.996435
dtype: float64

Các mức phân vị (Q1=25%, Q2=50%, Q3=75%):
      Quantity  UnitPrice  Sales
0.25       1.0       1.25   3.40
0.50       3.0       2.08   9.75
0.75      10.0       4.13  17.40


---
# PHẦN C — DEFINE THE QUESTION

## Câu hỏi 1: Quốc gia nào đóng góp doanh thu cao nhất, chiếm bao nhiêu % tổng doanh thu?

In [19]:
# TODO
country_sales = df.groupby('Country')['Sales'].sum().sort_values(ascending=False)
total_sales = df['Sales'].sum()
country_percent = (country_sales / total_sales) * 100

summary_country = pd.DataFrame({
    'Total Sales': country_sales,
    'Percentage (%)': country_percent
})
display(summary_country.head(10))

,Total Sales,Percentage (%)
Country,,
United Kingdom,8187806.364,83.996903
Netherlands,284661.540,2.920280
EIRE,263276.820,2.700899
Germany,221698.210,2.274353
France,197403.900,2.025123
Australia,137077.270,1.406246
Switzerland,56385.350,0.578445
Spain,54774.580,0.561920
Belgium,40910.960,0.419697


## Câu hỏi 2: Sản phẩm nào bán chạy nhất theo doanh thu?

In [22]:
# TODO
top_products = df.groupby(['StockCode', 'Description'])['Sales'].sum().sort_values(ascending=False)
display(top_products.head(10))

StockCode  Description                       
DOT        DOTCOM POSTAGE                        206245.48
22423      REGENCY CAKESTAND 3 TIER              164762.19
47566      PARTY BUNTING                          98302.98
85123A     WHITE HANGING HEART T-LIGHT HOLDER     97715.99
85099B     JUMBO BAG RED RETROSPOT                92356.03
23084      RABBIT NIGHT LIGHT                     66756.59
POST       POSTAGE                                66230.64
22086      PAPER CHAIN KIT 50'S CHRISTMAS         63791.94
84879      ASSORTED COLOUR BIRD ORNAMENT          58959.73
79321      CHILLI LIGHTS                          53768.06
Name: Sales, dtype: float64

## Câu hỏi 3: Doanh số có tính mùa vụ theo tháng không?

In [23]:
# TODO
df['Month'] = df['InvoiceDate'].dt.to_period('M')
monthly_sales = df.groupby('Month')['Sales'].sum()

print("Doanh số theo tháng:")
print(monthly_sales)

Doanh số theo tháng:
Month
2010-12     748957.020
2011-01     560000.260
2011-02     498062.650
2011-03     683267.080
2011-04     493207.121
2011-05     723333.510
2011-06     691123.120
2011-07     681300.111
2011-08     682680.510
2011-09    1019687.622
2011-10    1070704.670
2011-11    1461756.250
2011-12     433668.010
Freq: M, Name: Sales, dtype: float64


## Câu hỏi 4: Giá trị đơn hàng trung bình (Average Order Value) khác nhau thế nào giữa các quốc gia?

In [24]:
# TODO
invoice_totals = df.groupby(['Country', 'InvoiceNo'])['Sales'].sum().reset_index()

aov_by_country = invoice_totals.groupby('Country')['Sales'].mean().sort_values(ascending=False)
display(aov_by_country.head(10))

Country
Netherlands    2818.431089
Australia      1986.627101
Lebanon        1693.880000
Japan          1262.165000
Brazil         1143.600000
RSA            1002.310000
Singapore       912.039000
Denmark         893.720952
Norway          879.086500
Israel          878.646667
Name: Sales, dtype: float64

## Câu hỏi 5: Tỷ lệ giao dịch có dấu hiệu trả hàng/hủy (Quantity âm ở dữ liệu gốc) khác nhau thế nào giữa các quốc gia?

In [26]:
# TODO
df_raw = pd.read_csv(csv_path, encoding='ISO-8859-1', on_bad_lines='skip')
df_raw['Is_Cancel'] = df_raw['Quantity'] < 0

cancel_rate = df_raw.groupby('Country')['Is_Cancel'].mean().sort_values(ascending=False) * 100
display(cancel_rate.head(10))

Country
USA               38.487973
Czech Republic    16.666667
Malta             11.811024
Japan             10.335196
Saudi Arabia      10.000000
Australia          5.877681
Italy              5.603985
Bahrain            5.263158
Germany            4.770932
EIRE               3.684724
Name: Is_Cancel, dtype: float64

## Câu hỏi 6 (Tổng hợp) — Viết insight tổng hợp
Dựa trên Phần A, B, C, viết 4-5 câu insight tổng thể.

Thị trường United Kingdom (Anh) đóng góp tỷ trọng doanh thu áp đảo nhất, chiếm phần lớn tổng doanh thu toàn hệ thống.  
Doanh thu có xu hướng tăng mạnh vào những tháng cuối năm (đặc biệt là quý 4, từ tháng 9 đến tháng 11), thể hiện rõ tính mùa vụ mua sắm lễ hội.  
Giá trị đơn hàng trung bình (AOV) có sự chênh lệch lớn giữa các quốc gia, trong đó một số thị trường quốc tế có giá trị đơn hàng trung bình cao hơn mặc dù số lượng giao dịch ít hơn.  
Tỷ lệ đơn hàng bị hủy/trả lại xuất hiện ở hầu hết các quốc gia nhưng tập trung đáng kể tại một số thị trường chính, phản ánh tỷ lệ hoàn trả sản phẩm trong thương mại điện tử.